# Chapter 20a — Zero-Copy & Multi-GPU (companion)

> Companion to **Chapter 20 — Multi-GPU NCCL & ZeRO**, distilled from
> *CUDA by Example* (Sanders & Kandrot), **Chapter 11 — CUDA C on Multiple GPUs**.
> The single-GPU mechanics under data-parallel training.

Chapter 20 trained GPT-2 across multiple GPUs with NCCL. This companion builds the two primitives underneath that, on hardware you have:

1. **Zero-copy memory** — pinned host memory that the GPU reads/writes *directly over PCIe*, with no explicit `cudaMemcpy`. Useful in narrow cases; a trap in others.
2. **Multi-GPU work splitting** — `cudaSetDevice`, per-device allocations, and portable pinned memory, plus the rule that each GPU owns its own address space.

> **This is a single-GPU box (one RTX 4080 SUPER).** The zero-copy demo runs fully. The multi-GPU demo is written to split across *whatever devices exist* — so here it runs correctly on 1 GPU (a degenerate split), and the same code parallelizes across N GPUs on a bigger machine. Where a section truly needs ≥2 GPUs, it's flagged.

### Learning objectives

By the end you will:

- Allocate **mapped (zero-copy)** memory with `cudaHostAllocMapped` + `cudaHostGetDevicePointer`, and say when it helps vs. hurts.
- Split a workload across `cudaGetDeviceCount()` GPUs with `cudaSetDevice`, and explain why a pointer from one device can't be dereferenced on another.
- Connect both to Chapter 20's data-parallel training and NCCL.


## 1. Concept — Zero-Copy (Mapped) Memory

Normally you `cudaMalloc` device memory and `cudaMemcpy` into it. **Zero-copy** skips the copy: you allocate *mapped* pinned host memory and hand the kernel a device pointer that aliases it. Reads/writes from the kernel then travel over PCIe to host RAM on demand.

```c
cudaSetDeviceFlags(cudaDeviceMapHost);                 // must precede context creation
float* h;  cudaHostAlloc(&h, bytes, cudaHostAllocMapped | cudaHostAllocPortable);
float* d;  cudaHostGetDevicePointer(&d, h, 0);          // device alias of the same memory
kernel<<<grid, block>>>(d, ...);                        // GPU touches host RAM directly
cudaDeviceSynchronize();                                // h now holds the results — no memcpy
```

When is this a *win*?

- **Integrated GPUs** (laptop/Jetson) where "device memory" *is* system memory — the copy was pure waste.
- **Read-once / write-once** data: if each byte is touched exactly once, the PCIe transfer is unavoidable anyway, and zero-copy overlaps it with compute for free.

When is it a *trap*? **Repeated access.** Every access to zero-copy memory crosses PCIe (~tens of GB/s) instead of hitting device DRAM (~700 GB/s). A kernel that reads an array many times over zero-copy memory will crawl. For `llm.c`'s weights — read every layer, every step — zero-copy would be a disaster; they live in device memory.


In [ ]:
!mkdir -p course/ch20a_build


In [ ]:
%%writefile course/ch20a_build/zero_copy.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] + b[i];     // each element touched once -> good zero-copy case
}

int main(void) {
    cudaDeviceProp prop; cudaGetDeviceProperties(&prop, 0);
    if (!prop.canMapHostMemory) { printf("device cannot map host memory\n"); return 0; }
    cudaSetDeviceFlags(cudaDeviceMapHost);

    const int N = 1 << 22;
    float *a, *b, *c;                  // host pointers (mapped, pinned)
    cudaHostAlloc(&a, N*sizeof(float), cudaHostAllocMapped | cudaHostAllocPortable);
    cudaHostAlloc(&b, N*sizeof(float), cudaHostAllocMapped | cudaHostAllocPortable);
    cudaHostAlloc(&c, N*sizeof(float), cudaHostAllocMapped | cudaHostAllocPortable);
    for (int i = 0; i < N; i++) { a[i] = i; b[i] = 2.0f*i; }

    float *da, *db, *dc;               // device aliases of the same memory
    cudaHostGetDevicePointer(&da, a, 0);
    cudaHostGetDevicePointer(&db, b, 0);
    cudaHostGetDevicePointer(&dc, c, 0);

    int block = 256, grid = (N + block - 1) / block;
    vadd<<<grid, block>>>(da, db, dc, N);
    cudaDeviceSynchronize();           // no cudaMemcpy needed: c is already host-visible

    int ok = 1;
    for (int i = 0; i < N; i++) if (c[i] != 3.0f*i) { ok = 0; break; }
    printf("zero-copy vadd: %s  (c[0..3] = %.0f %.0f %.0f %.0f)\n",
           ok ? "PASS" : "FAIL", c[0], c[1], c[2], c[3]);

    cudaFreeHost(a); cudaFreeHost(b); cudaFreeHost(c);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20a_build/zero_copy course/ch20a_build/zero_copy.cu && ./course/ch20a_build/zero_copy


`PASS` — and notice there is **no `cudaMemcpy` anywhere**. The kernel wrote its results straight into host-visible memory. For this once-through vector add that's fine. If `vadd` instead looped over `a` and `b` a hundred times, this version would be dramatically slower than a `cudaMalloc` + copy version, because every one of those reads would re-cross PCIe.


## 2. Concept — Multiple GPUs

Each GPU is a separate device with its **own** memory and its **own** address space. The rules:

- `cudaGetDeviceCount(&n)` tells you how many GPUs there are.
- `cudaSetDevice(d)` makes device `d` *current* for the calling host thread — subsequent `cudaMalloc`, kernel launches, and copies target that device.
- A device pointer from GPU 0 is **meaningless on GPU 1**. You can't pass `d_a` allocated on device 0 to a kernel on device 1; you must allocate per-device and copy (or enable peer-to-peer access).
- **Portable** pinned memory (`cudaHostAllocPortable`) is pinned for *all* devices/contexts, not just the one current when it was allocated — needed when several GPUs stream from the same host buffer.

The standard pattern: split the data into `n` slices, give slice `d` to device `d`, run, gather. On a real multi-GPU box the slices run **in parallel**. The code below splits across however many devices exist — so it's correct on 1 GPU (everything goes to device 0) and scales out unchanged.


In [ ]:
%%writefile course/ch20a_build/multigpu.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main(void) {
    int ndev = 0; cudaGetDeviceCount(&ndev);
    if (ndev < 1) { printf("no CUDA devices\n"); return 0; }
    printf("device count = %d%s\n", ndev,
           (ndev < 2) ? "  (single-GPU box: the split below runs all slices on device 0;"
                        " with >=2 GPUs they run in parallel)" : "");

    const int N = 1 << 22;
    float* ha = (float*)malloc(N*sizeof(float));
    float* hb = (float*)malloc(N*sizeof(float));
    float* hc = (float*)malloc(N*sizeof(float));
    for (int i = 0; i < N; i++) { ha[i] = i; hb[i] = 3.0f*i; }

    int per = (N + ndev - 1) / ndev;       // elements per device
    for (int d = 0; d < ndev; d++) {
        cudaSetDevice(d);                  // device d is now current for this thread
        int off = d * per;
        int len = (off + per <= N) ? per : (N - off);
        if (len <= 0) continue;
        float *da, *db, *dc;
        cudaMalloc(&da, len*sizeof(float));
        cudaMalloc(&db, len*sizeof(float));
        cudaMalloc(&dc, len*sizeof(float));
        cudaMemcpy(da, ha+off, len*sizeof(float), cudaMemcpyHostToDevice);
        cudaMemcpy(db, hb+off, len*sizeof(float), cudaMemcpyHostToDevice);
        int block = 256, grid = (len + block - 1) / block;
        vadd<<<grid, block>>>(da, db, dc, len);
        cudaMemcpy(hc+off, dc, len*sizeof(float), cudaMemcpyDeviceToHost);
        cudaFree(da); cudaFree(db); cudaFree(dc);
    }

    int ok = 1;
    for (int i = 0; i < N; i++) if (hc[i] != 4.0f*i) { ok = 0; break; }
    printf("multi-GPU split vadd: %s  (hc[N-1] = %.0f, expected %.0f)\n",
           ok ? "PASS" : "FAIL", hc[N-1], 4.0f*(N-1));

    free(ha); free(hb); free(hc);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20a_build/multigpu course/ch20a_build/multigpu.cu && ./course/ch20a_build/multigpu


On this box you'll see `device count = 1` and the split runs entirely on device 0 — still `PASS`. The same binary on a 4-GPU server would print `device count = 4` and run the four slices on four GPUs concurrently (with one host thread per GPU, or CUDA's async APIs, to actually overlap them).


## 3. Translation Bridge — From Splitting to Data-Parallel Training

| Book (Ch11) | `llm.c` Chapter 20 | Role |
|---|---|---|
| split array into `ndev` slices | shard the **token batch** across GPUs (data parallelism) | each GPU does the same model on different data |
| `cudaSetDevice(d)` per slice | one process per GPU (`mpirun -np N`), each pinned to its device | own the device |
| gather partial results on host | **NCCL all-reduce** of gradients across GPUs | average grads so every replica steps identically |
| portable pinned staging | pinned buffers for NCCL / host↔device | fast, async-capable transfers |
| (not in book) | **ZeRO** shards optimizer state across GPUs | cut per-GPU memory |

The leap from this companion to Chapter 20 is *what* you communicate and *how*: instead of the host gathering result slices, the GPUs themselves all-reduce gradients directly (over NVLink/PCIe/network) via **NCCL**, overlapping that communication with the backward pass (Appendix 02's overlap idea). Zero-copy, by contrast, barely appears in training — weights are accessed far too often to live across PCIe.


## 4. Common Pitfalls

- **`cudaSetDeviceFlags(cudaDeviceMapHost)` must come before any context** (before the first runtime call that creates one). Set it too late and `cudaHostGetDevicePointer` fails.
- **Zero-copy for hot data is a performance bug** — every access crosses PCIe. Use it only for read-once/write-once or integrated GPUs.
- **Cross-device pointer use** — a `cudaMalloc` pointer from device 0 is invalid on device 1. Allocate per device; use P2P or explicit copies to move data.
- **`cudaSetDevice` is per host thread** — multi-GPU code typically uses one thread (or process) per GPU; forgetting to set the device leaves work on device 0.
- **Non-portable pinned memory** is only pinned for the context that allocated it; add `cudaHostAllocPortable` when multiple GPUs share the buffer.


## 5. TODO Exercise — Convert to Zero-Copy

The vector add below uses the classic `cudaMalloc` + `cudaMemcpy` flow. Convert it to **zero-copy**: allocate mapped host memory, get device pointers, drop the explicit copies. Fill in the TODOs.


In [ ]:
%%writefile course/ch20a_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void scale(const float* a, float* b, float k, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) b[i] = k * a[i];
}

int main(void) {
    const int N = 1 << 20;
    // TODO 1: cudaSetDeviceFlags(cudaDeviceMapHost);
    float *a, *b;
    // TODO 2: allocate a and b with cudaHostAlloc(..., cudaHostAllocMapped)
    a = (float*)malloc(N*sizeof(float));   // <-- replace with mapped alloc
    b = (float*)malloc(N*sizeof(float));   // <-- replace with mapped alloc
    for (int i = 0; i < N; i++) a[i] = i;

    float *da, *db;
    // TODO 3: get device pointers via cudaHostGetDevicePointer for a and b
    cudaMalloc(&da, N*sizeof(float));      // <-- replace with device-pointer-of-a
    cudaMalloc(&db, N*sizeof(float));      // <-- replace with device-pointer-of-b
    cudaMemcpy(da, a, N*sizeof(float), cudaMemcpyHostToDevice);  // <-- not needed with zero-copy

    int block = 256, grid = (N + block - 1) / block;
    scale<<<grid, block>>>(da, db, 2.0f, N);
    cudaDeviceSynchronize();
    cudaMemcpy(b, db, N*sizeof(float), cudaMemcpyDeviceToHost);  // <-- not needed with zero-copy

    int ok = 1; for (int i = 0; i < N; i++) if (b[i] != 2.0f*i) { ok = 0; break; }
    printf("%s  (b[0..3] = %.0f %.0f %.0f %.0f)\n", ok ? "PASS" : "FAIL", b[0], b[1], b[2], b[3]);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20a_build/exercise1 course/ch20a_build/exercise1.cu && ./course/ch20a_build/exercise1


### Solution

In [ ]:
%%writefile course/ch20a_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void scale(const float* a, float* b, float k, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) b[i] = k * a[i];
}

int main(void) {
    const int N = 1 << 20;
    cudaSetDeviceFlags(cudaDeviceMapHost);                       // TODO 1
    float *a, *b;
    cudaHostAlloc(&a, N*sizeof(float), cudaHostAllocMapped);     // TODO 2
    cudaHostAlloc(&b, N*sizeof(float), cudaHostAllocMapped);
    for (int i = 0; i < N; i++) a[i] = i;

    float *da, *db;
    cudaHostGetDevicePointer(&da, a, 0);                         // TODO 3
    cudaHostGetDevicePointer(&db, b, 0);
    // no host->device copy needed

    int block = 256, grid = (N + block - 1) / block;
    scale<<<grid, block>>>(da, db, 2.0f, N);
    cudaDeviceSynchronize();                                     // b is already host-visible

    int ok = 1; for (int i = 0; i < N; i++) if (b[i] != 2.0f*i) { ok = 0; break; }
    printf("%s  (b[0..3] = %.0f %.0f %.0f %.0f)\n", ok ? "PASS" : "FAIL", b[0], b[1], b[2], b[3]);
    cudaFreeHost(a); cudaFreeHost(b);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20a_build/exercise1_sol course/ch20a_build/exercise1_sol.cu && ./course/ch20a_build/exercise1_sol


## Recap

- **Zero-copy** (mapped pinned memory + `cudaHostGetDevicePointer`) lets the GPU touch host RAM directly with no `cudaMemcpy` — great for read-once data and integrated GPUs, terrible for hot data (every access crosses PCIe).
- **Multiple GPUs** each have their own memory and address space: `cudaGetDeviceCount`, `cudaSetDevice`, per-device allocations, and `cudaHostAllocPortable` for shared host buffers.
- Splitting an array across `ndev` devices is the seed of **data-parallel training**; the real difference in Chapter 20 is that GPUs all-reduce gradients with **NCCL** instead of the host gathering slices.

### What's next

You've now reconstructed the single-GPU mechanics under Chapter 20's multi-GPU training. The optional **Appendix 03 — Advanced Atomics** (book Appendix A) closes the loop on the atomics from Chapter 13b with compare-and-swap locks. Otherwise: you've walked the whole GPU half of the course, book companion by book companion. 🎉
